In [1]:
%pip install pycryptodome

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import json
from Crypto.Cipher import AES, PKCS1_OAEP
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes

# 1. Generate Kunci RSA
rsa_key = RSA.generate(2048)
public_key = rsa_key.public_key().export_key()

# 2. Fungsi Inti Enkripsi
def encrypt_laporan(data_dict, pub_key_str):
    data_bytes = json.dumps(data_dict).encode('utf-8')
    aes_key = get_random_bytes(16)
    cipher_aes = AES.new(aes_key, AES.MODE_GCM)
    ciphertext, tag = cipher_aes.encrypt_and_digest(data_bytes)
    
    recipient_key = RSA.import_key(pub_key_str)
    cipher_rsa = PKCS1_OAEP.new(recipient_key)
    enc_aes_key = cipher_rsa.encrypt(aes_key)
    
    return {
        "encrypted_data": ciphertext.hex(),
        "encrypted_aes_key": enc_aes_key.hex(),
        "nonce": cipher_aes.nonce.hex(),
        "tag": tag.hex()
    }

print("Kunci RSA, dan Fungsi Enkripsi sudah siap!")

Kunci RSA, dan Fungsi Enkripsi sudah siap!


In [3]:
# 1. Baca dataset utuh
df = pd.read_csv('test_data_with_predictions.csv')

# 2. Siapkan list kosong sebagai "keranjang" data terenkripsi
semua_data_terenkripsi = []

print(f"Memulai proses enkripsi untuk {len(df)} baris data... Mohon tunggu sebentar.")

# 3. Looping dari baris pertama hingga terakhir
for index, row in df.iterrows():
    
    # Tentukan label status
    status_label = "Normal"
    if row['Predicted_Label'] == 1:
        status_label = "Attack"
    elif row['Predicted_Label'] == 2:
        status_label = "Fault"
        
    # Buat format laporan
    laporan = {
        "timestamp": str(row['timestamp']),
        "device_id": str(row['device_id']),
        "voltage": float(row['voltage']),
        "latency": float(row['latency']),
        "status": status_label
    }
    
    # Enkripsi dan masukkan ke keranjang
    paket_aman = encrypt_laporan(laporan, public_key)
    semua_data_terenkripsi.append(paket_aman)

print("Seluruh baris data berhasil dikunci!")

Memulai proses enkripsi untuk 4999 baris data... Mohon tunggu sebentar.
Seluruh baris data berhasil dikunci!


In [4]:
# Menampilkan 3 data pertama sebagai bukti
print("=== PREVIEW 3 BRANKAS PERTAMA ===")
for i in range(3):
    print(f"--- Data Baris ke-{i+1} ---")
    print(f"Ciphertext : {semua_data_terenkripsi[i]['encrypted_data'][:40]}... (disembunyikan)")
    print(f"Kunci AES  : {semua_data_terenkripsi[i]['encrypted_aes_key'][:40]}... (disembunyikan)")
    print(f"Tag (MAC)  : {semua_data_terenkripsi[i]['tag']}\n")

=== PREVIEW 3 BRANKAS PERTAMA ===
--- Data Baris ke-1 ---
Ciphertext : ee8ebc2d407a69da6257f77c747a329ef10f1a5a... (disembunyikan)
Kunci AES  : 4a9285452798e00cab79d61dcc3165c0046d0d5b... (disembunyikan)
Tag (MAC)  : 7818b86ae90349c2c7c60e6187a2e500

--- Data Baris ke-2 ---
Ciphertext : 98f9db21ee422586c63289561c4a6915e53ff002... (disembunyikan)
Kunci AES  : 59486398c973eb7283ebb925324ddd623d7a20b7... (disembunyikan)
Tag (MAC)  : 2791c9ab47708683f8bdabe1cd3e651d

--- Data Baris ke-3 ---
Ciphertext : 39ea64ae5fa76cd03a4778e565cd34ebd18b024d... (disembunyikan)
Kunci AES  : 46f0fca57b1a40975e8cad3110cd593f47216d52... (disembunyikan)
Tag (MAC)  : 5b530411600964913e7dfa100a8ee71c

